In [1]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [32]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
import torch_geometric
from torch_geometric.data import Data
from torch_geometric_temporal.signal import StaticGraphTemporalSignal, temporal_signal_split
from torch.utils.data import IterableDataset, DataLoader
from torch_geometric.nn import SAGEConv, GATConv, GeneralConv, TransformerConv

In [33]:
class MLP(nn.Module):
    def __init__(self, in_channels, hidden_layers, out_channels):
        super().__init__()
        layers = []
        prev_layer = in_channels

        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(prev_layer, hidden_dim))
            layers.append(nn.ReLU())
            prev_layer = hidden_dim

        layers.append(nn.Linear(prev_layer, out_channels))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(x)

# Define the GRUDecoder class
class GRUDecoder(nn.Module):
    def __init__(self, in_channels, out_channels, hidden_gru=64):
        super().__init__()
        hidden_layers = [128, 64, 32]

        self.gru = nn.GRU(in_channels, hidden_gru, batch_first=True)
        self.mlp = MLP(hidden_gru, hidden_layers, out_channels)

    def forward(self, z):
        gru_out, _ = self.gru(z)
        output = self.mlp(gru_out.squeeze())
        return output

# Define the GNNEncoder classes
class GNNEncoder1(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)

    def forward(self, x, edge_index, edge_attr=None):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

class GNNEncoder2(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GATConv((-1, -1), hidden_channels, add_self_loops=False)
        self.conv2 = GATConv((-1, -1), out_channels, add_self_loops=False)

    def forward(self, x, edge_index, edge_attr):
        x = self.conv1(x, edge_index, edge_attr).relu()
        x = self.conv2(x, edge_index, edge_attr)
        return x

class GNNEncoder3(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GeneralConv((-1, -1), hidden_channels, in_edge_channels=-1)
        self.conv2 = GeneralConv((-1, -1), out_channels, in_edge_channels=-1)

    def forward(self, x, edge_index, edge_attr):
        x = self.conv1(x, edge_index, edge_attr).relu()
        x = self.conv2(x, edge_index, edge_attr)
        return x

class GNNEncoder4(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=1, dropout=0.0, edge_dim=None):
        super().__init__()
        self.conv1 = TransformerConv(
            in_channels=in_channels,
            out_channels=hidden_channels,
            heads=heads,
            concat=True,
            beta=False,
            dropout=dropout,
            edge_dim=edge_dim,
            bias=True,
            root_weight=True
        )
        self.conv2 = TransformerConv(
            in_channels=hidden_channels * heads,
            out_channels=out_channels,
            heads=heads,
            concat=True,
            beta=False,
            dropout=dropout,
            edge_dim=edge_dim,
            bias=True,
            root_weight=True
        )

    def forward(self, x, edge_index, edge_attr=None):
        x = self.conv1(x, edge_index, edge_attr).relu()
        x = self.conv2(x, edge_index, edge_attr)
        return x


In [34]:
class Model(torch.nn.Module):
    def __init__(self, input_size, hidden_channels, out_channels, edge_index, edge_attr=None, encoder_type='GNNEncoder4'):
        super().__init__()
        if encoder_type == 'GNNEncoder1':
            self.encoder = GNNEncoder1(hidden_channels, hidden_channels)
        elif encoder_type == 'GNNEncoder2':
            self.encoder = GNNEncoder2(hidden_channels, hidden_channels)
        elif encoder_type == 'GNNEncoder3':
            self.encoder = GNNEncoder3(hidden_channels, hidden_channels)
        elif encoder_type == 'GNNEncoder4':
            self.encoder = GNNEncoder4(input_size, hidden_channels, hidden_channels, edge_dim=edge_attr.size(1) if edge_attr is not None else None)
        else:
            raise ValueError(f"Unknown encoder type: {encoder_type}")

        self.decoder = GRUDecoder(hidden_channels, out_channels)
        self.edge_index = edge_index
        self.edge_attr = edge_attr

    def forward(self, x):
        z = self.encoder(x, self.edge_index, self.edge_attr)
        z = z.unsqueeze(1)
        return self.decoder(z)


In [35]:
class CustomIterableDataset(IterableDataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __iter__(self):
        for data in self.dataset:
            yield data

# Load the dataset
try:
    speed_data = pd.read_csv('dataset/PeMSD7_V_228.csv', header=None)
    adjacency_matrix = pd.read_csv('dataset/PeMSD7_W_228.csv', header=None)
except FileNotFoundError:
    import os
    import requests
    import zipfile

    url = "https://github.com/VeritasYin/STGCN_IJCAI-18/raw/master/dataset/PeMSD7_Full.zip"
    os.makedirs('dataset', exist_ok=True)
    response = requests.get(url)
    zip_path = 'dataset/PeMSD7_Full.zip'
    with open(zip_path, 'wb') as f:
        f.write(response.content)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('dataset')

    speed_data = pd.read_csv('dataset/PeMSD7_V_228.csv', header=None)
    adjacency_matrix = pd.read_csv('dataset/PeMSD7_W_228.csv', header=None)


In [36]:
scaler = StandardScaler()
speed_data = scaler.fit_transform(speed_data)

train_size = int(len(speed_data) * 0.8)
train_data = speed_data[:train_size]
test_data = speed_data[train_size:]

train_data = torch.tensor(train_data, dtype=torch.float32)
test_data = torch.tensor(test_data, dtype=torch.float32)

print(train_data.shape)
print(test_data.shape)

torch.Size([10137, 228])
torch.Size([2535, 228])


In [37]:
adjacency_matrix = adjacency_matrix.values  # Convert to numpy array
edge_index = torch.tensor(np.stack(np.nonzero(adjacency_matrix)), dtype=torch.long)
edge_attr = adjacency_matrix[edge_index[0], edge_index[1]]
edge_attr = torch.tensor(edge_attr, dtype=torch.float).unsqueeze(1)  # Shape: [num_edges, 1]


In [38]:
#graph object
graph = Data(edge_index=edge_index, edge_attr=edge_attr)

print(graph.edge_index)
print(graph.edge_attr)

tensor([[  0,   0,   0,  ..., 227, 227, 227],
        [  1,   2,   3,  ..., 223, 225, 226]])
tensor([[ 3165.9399],
        [ 8731.5400],
        [11903.4502],
        ...,
        [12631.4600],
        [13725.7900],
        [17550.3594]])


In [39]:
dataset = StaticGraphTemporalSignal(
    edge_index=graph.edge_index,
    edge_weight=graph.edge_attr,
    features=train_data[:-1],  # Use the same temporal dimension for features
    targets=train_data[1:]     # Ensure targets match the temporal dimension of features
)

In [40]:
train_dataset, test_dataset = temporal_signal_split(dataset, train_ratio=0.8)

In [41]:
train_dataset = CustomIterableDataset(train_dataset)
test_dataset = CustomIterableDataset(test_dataset)

# Create DataLoader for training and test datasets
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)  # shuffle=False for iterable datasets
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


input_size = speed_data.shape[1]
hidden_channels = input_size
out_channels = speed_data.shape[1]
num_epochs = 50
learning_rate = 0.001


encoder_type = 'GNNEncoder1'
model = Model(input_size, hidden_channels, out_channels, graph.edge_index, edge_attr=None, encoder_type=encoder_type)
optimizer = Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()


In [42]:
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        # Prepare input and target
        input_data, target_data = batch.features, batch.targets
        # Forward pass
        output = model(input_data)
        loss = loss_fn(output, target_data)
        # Backward pass
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(train_loader)}')

AttributeError: 'torch.dtype' object has no attribute 'kind'

In [43]:
model.eval()
total_mse = 0
total_mae = 0
total_mape = 0
with torch.no_grad():
    for batch in test_loader:
        # Prepare input and target
        input_data, target_data = batch.features, batch.targets
        # Forward pass
        output = model(input_data)
        # Calculate metrics
        mse = mean_squared_error(target_data.numpy(), output.numpy())
        mae = mean_absolute_error(target_data.numpy(), output.numpy())
        mape = mean_absolute_percentage_error(target_data.numpy(), output.numpy())
        total_mse += mse
        total_mae += mae
        total_mape += mape

# Calculate average metrics
avg_mse = total_mse / len(test_loader)
avg_mae = total_mae / len(test_loader)
avg_mape = total_mape / len(test_loader)
avg_rmse = np.sqrt(avg_mse)

print(f'MSE: {avg_mse}')
print(f'MAE: {avg_mae}')
print(f'MAPE: {avg_mape}')
print(f'RMSE: {avg_rmse}')

AttributeError: 'torch.dtype' object has no attribute 'kind'